# Ablation Benchmark Selector — LocalStack-deployable subset for the IaCGOD ablation study

## §0 Configuration

In [1]:
import hashlib
import json
import os
import random
import re
import shutil
import subprocess
import tempfile
import time
import uuid
from pathlib import Path

import numpy as np
import pandas as pd
import requests

DATA_ANALYSIS_DIR = Path('.').resolve()
IACGOD_DATA_DIR    = (DATA_ANALYSIS_DIR / '../data').resolve()

CFN_BASE_DIR       = DATA_ANALYSIS_DIR / 'cfn_benchmark'
CFN_DATASET_DIR    = CFN_BASE_DIR / 'dataset'
CFN_WITH_PROMPTS   = CFN_DATASET_DIR / 'final_benchmark_real_aws_with_prompts.csv'
CFN_LOCALSTACK_CACHE = CFN_DATASET_DIR / 'deploy_cache_aws_stack.csv'
CFN_EVAL_CSV       = IACGOD_DATA_DIR / 'cfn_eval_benchmark_real_aws.csv'

TF_BASE_DIR        = DATA_ANALYSIS_DIR / 'iac_benchmark'
TF_DATASET_DIR     = TF_BASE_DIR / 'dataset'
TF_WITH_PROMPTS    = TF_DATASET_DIR / 'final_benchmark_real_aws_with_prompts.csv'
TF_LOCALSTACK_CACHE = TF_DATASET_DIR / 'deploy_cache.csv'
TF_PLUGIN_CACHE_DIR = (TF_BASE_DIR / '.tf_plugin_cache').resolve()
TF_EVAL_CSV        = IACGOD_DATA_DIR / 'tf_eval_benchmark_real_aws.csv'

OUT_DIR = DATA_ANALYSIS_DIR / 'ablation_selection'
OUT_DIR.mkdir(exist_ok=True)
SELECTION_CSV       = OUT_DIR / 'ablation_benchmark_selection.csv'
CFN_ABLATION_EVAL   = OUT_DIR / 'cfn_eval_benchmark_ablation.csv'
TF_ABLATION_EVAL    = OUT_DIR / 'tf_eval_benchmark_ablation.csv'
CFN_ABLATION_EVAL_IACGOD = IACGOD_DATA_DIR / 'cfn_eval_benchmark_ablation.csv'
TF_ABLATION_EVAL_IACGOD  = IACGOD_DATA_DIR / 'tf_eval_benchmark_ablation.csv'

TOTAL_SCENARIOS  = 70
CFN_FRACTION     = 0.5
CFN_TARGET       = round(TOTAL_SCENARIOS * CFN_FRACTION)
TF_TARGET        = TOTAL_SCENARIOS - CFN_TARGET

LEVEL_WEIGHTS    = {1: 1, 2: 2, 3: 3, 4: 4, 5: 5}
OVERSAMPLE_FACTOR = 1.6
RNG_SEED         = 42

FREEZE_ORIGINAL_SELECTION = True

RUN_LOCALSTACK_CHECK_CFN = True
RUN_LOCALSTACK_CHECK_TF  = True
RECHECK_TF_CONFIRMED     = True
LOCALSTACK_ENDPOINT      = os.environ.get('LOCALSTACK_ENDPOINT', 'http://localhost:4566')
DEPLOY_TIMEOUT_MIN       = 15
AWS_REGION               = 'us-east-1'
AWS_STACK_PREFIX         = 'ablation-check'

print(f"CFN target: {CFN_TARGET}  |  TF target: {TF_TARGET}  |  total: {CFN_TARGET + TF_TARGET}")
print(f"Level weights: {LEVEL_WEIGHTS}")

CFN target: 35  |  TF target: 35  |  total: 70
Level weights: {1: 1, 2: 2, 3: 3, 4: 4, 5: 5}


## §1 Load the frozen real-AWS benchmarks

In [2]:
df_cfn = pd.read_csv(CFN_WITH_PROMPTS, low_memory=False)
cfn_eval = pd.read_csv(CFN_EVAL_CSV)

_merged = cfn_eval.merge(df_cfn, left_on='ground_truth_path', right_on='dest_file',
                          how='left', suffixes=('_eval', ''))
_missing = _merged['dest_file'].isna().sum()
_diff_mismatch = (_merged['difficulty_eval'] != _merged['difficulty']).sum()
_prompt_mismatch = (_merged['prompt'] != _merged['user_prompt']).sum()
print(f"CFN: {len(cfn_eval)} eval rows, {len(df_cfn)} with_prompts rows -- "
      f"{_missing} unmatched, {_diff_mismatch} difficulty mismatches, {_prompt_mismatch} prompt mismatches")
assert _missing == 0 and _diff_mismatch == 0 and _prompt_mismatch == 0, \
    'CFN eval CSV has drifted from final_benchmark_real_aws_with_prompts.csv -- investigate before selecting'

df_cfn = df_cfn.copy()
df_cfn['lang'] = 'cfn'
df_cfn['path_key'] = df_cfn['dest_file']
df_cfn['diversity_key'] = df_cfn['aws_services'].fillna('UNKNOWN')

df_tf = pd.read_csv(TF_WITH_PROMPTS, low_memory=False)
tf_eval = pd.read_csv(TF_EVAL_CSV)

_mergedt = tf_eval.merge(df_tf, left_on='ground_truth_path', right_on='folder_path',
                          how='left', suffixes=('_eval', ''))
_missingt = _mergedt['folder_path'].isna().sum()
_diff_mismatcht = (_mergedt['difficulty_eval'] != _mergedt['difficulty']).sum()
_prompt_mismatcht = (_mergedt['prompt'] != _mergedt['user_prompt']).sum()
print(f"TF:  {len(tf_eval)} eval rows, {len(df_tf)} with_prompts rows -- "
      f"{_missingt} unmatched, {_diff_mismatcht} difficulty mismatches, {_prompt_mismatcht} prompt mismatches")
assert _missingt == 0 and _diff_mismatcht == 0 and _prompt_mismatcht == 0, \
    'TF eval CSV has drifted from final_benchmark_real_aws_with_prompts.csv -- investigate before selecting'

df_tf = df_tf.copy()
df_tf['lang'] = 'tf'
df_tf['path_key'] = df_tf['folder_path']
df_tf['diversity_key'] = df_tf['aws_services'].fillna('UNKNOWN')

print(f"\nCFN rows: {len(df_cfn)} (levels {sorted(df_cfn['difficulty'].unique())}), "
      f"TF rows: {len(df_tf)} (levels {sorted(df_tf['difficulty'].unique())})")

CFN: 250 eval rows, 250 with_prompts rows -- 0 unmatched, 0 difficulty mismatches, 0 prompt mismatches
TF:  250 eval rows, 250 with_prompts rows -- 0 unmatched, 0 difficulty mismatches, 0 prompt mismatches

CFN rows: 250 (levels [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]), TF rows: 250 (levels [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)])


## §2 LocalStack prior-pass signal

In [3]:
def _normalise_cfn(text: str) -> str:
    text = text.replace('\r\n', '\n').replace('\r', '\n')
    text = text.replace('\t', '  ')
    text = '\n'.join(line.rstrip() for line in text.split('\n'))
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()


def _fresh_cfn_hash(dest_file: str):
    p = CFN_BASE_DIR / dest_file
    if not p.exists():
        return None
    text = p.read_text(encoding='utf-8', errors='ignore')
    return hashlib.sha256(_normalise_cfn(text).encode('utf-8')).hexdigest()


df_cfn['fresh_content_hash'] = df_cfn['dest_file'].map(_fresh_cfn_hash)
_missing_files = df_cfn['fresh_content_hash'].isna().sum()
if _missing_files:
    print(f"⚠️ {_missing_files} CFN ground-truth file(s) missing on disk -- treated as ls_status=None")

_ls_cfn = pd.read_csv(CFN_LOCALSTACK_CACHE, usecols=['content_hash', 'deploy_pass'], low_memory=False)
_ls_cfn = _ls_cfn.dropna(subset=['content_hash']).drop_duplicates('content_hash', keep='last')
_cfn_status_map = dict(zip(_ls_cfn['content_hash'], _ls_cfn['deploy_pass']))
df_cfn['ls_status'] = df_cfn['fresh_content_hash'].map(_cfn_status_map)
df_cfn['ls_status'] = df_cfn['ls_status'].map(lambda v: bool(v) if pd.notna(v) else None)

print('CFN prior LocalStack status (content-hash-fresh):')
print(df_cfn['ls_status'].value_counts(dropna=False))
print()
print(df_cfn.groupby('difficulty')['ls_status'].value_counts(dropna=False).unstack(fill_value=0))

CFN prior LocalStack status (content-hash-fresh):
ls_status
True     215
None      34
False      1
Name: count, dtype: int64

ls_status   True  NaN  False
difficulty                  
1             44    6      0
2             45    5      0
3             42    8      0
4             43    7      0
5             41    8      1


In [4]:
def _base_scenario_id(sid: str) -> str:
    return sid[:-len('_manualfix')] if sid.endswith('_manualfix') else sid


_dc_tf = pd.read_csv(TF_LOCALSTACK_CACHE, usecols=['scenario_id', 'deploy_passed'], low_memory=False)
_dc_tf = _dc_tf.dropna(subset=['scenario_id'])

_tf_exact_map = {}
for _sid, _passed in zip(_dc_tf['scenario_id'], _dc_tf['deploy_passed']):
    if _sid not in _tf_exact_map or bool(_passed):
        _tf_exact_map[_sid] = bool(_passed) if (_sid not in _tf_exact_map or _passed) else _tf_exact_map[_sid]

_tf_base_map = {}
for _sid, _passed in zip(_dc_tf['scenario_id'], _dc_tf['deploy_passed']):
    _b = _base_scenario_id(_sid)
    if _b not in _tf_base_map or bool(_passed):
        _tf_base_map[_b] = bool(_passed) if (_b not in _tf_base_map or _passed) else _tf_base_map[_b]


def _tf_lookup(sid: str):
    if sid in _tf_exact_map:
        return _tf_exact_map[sid]
    b = _base_scenario_id(sid)
    if b in _tf_exact_map:
        return _tf_exact_map[b]
    return _tf_base_map.get(b)


df_tf['ls_status'] = df_tf['scenario_id'].map(_tf_lookup)

print('TF prior LocalStack status (scenario_id / base-id match -- SOFT signal, not content-fresh-verified):')
print(df_tf['ls_status'].value_counts(dropna=False))
print()
print(df_tf.groupby('difficulty')['ls_status'].value_counts(dropna=False).unstack(fill_value=0))

TF prior LocalStack status (scenario_id / base-id match -- SOFT signal, not content-fresh-verified):
ls_status
True     154
None      53
False     43
Name: count, dtype: int64

ls_status   False  True  NaN
difficulty                  
1               2    32   16
2               4    33   13
3               3    33   14
4               8    32   10
5              26    24    0


## §3 Level-weighted target allocation

In [5]:
def allocate_targets(total: int, weights_by_level: dict) -> dict:
    levels = sorted(weights_by_level)
    wsum = sum(weights_by_level[l] for l in levels)
    raw = {l: total * weights_by_level[l] / wsum for l in levels}
    floors = {l: int(raw[l]) for l in levels}
    remainder = total - sum(floors.values())
    frac_order = sorted(levels, key=lambda l: raw[l] - floors[l], reverse=True)
    for l in frac_order[:remainder]:
        floors[l] += 1
    assert sum(floors.values()) == total
    return floors


cfn_targets = allocate_targets(CFN_TARGET, LEVEL_WEIGHTS)
tf_targets = allocate_targets(TF_TARGET, LEVEL_WEIGHTS)

print('CFN per-level targets:', cfn_targets, ' (sum =', sum(cfn_targets.values()), ')')
print('TF  per-level targets:', tf_targets, ' (sum =', sum(tf_targets.values()), ')')

CFN per-level targets: {1: 2, 2: 5, 3: 7, 4: 9, 5: 12}  (sum = 35 )
TF  per-level targets: {1: 2, 2: 5, 3: 7, 4: 9, 5: 12}  (sum = 35 )


## §4 Diversity-aware, status-tiered ranking

In [6]:
def _status_tier(status) -> int:
    if status is True:
        return 0
    if status is False:
        return 2
    return 1


def _diverse_round_robin(rows: list, seed: int) -> list:
    rng = random.Random(seed)
    groups = {}
    for row in rows:
        groups.setdefault(row['diversity_key'], []).append(row)
    group_keys = list(groups.keys())
    rng.shuffle(group_keys)
    ordered = []
    idx = 0
    remaining = len(rows)
    while remaining > 0:
        key = group_keys[idx % len(group_keys)]
        if groups[key]:
            ordered.append(groups[key].pop(0))
            remaining -= 1
        idx += 1
    return ordered


def rank_level_candidates(df_level: pd.DataFrame, seed: int) -> list:
    rows = [row for _, row in df_level.iterrows()]
    tiers = {0: [], 1: [], 2: []}
    for row in rows:
        tiers[_status_tier(row['ls_status'])].append(row)
    ordered = []
    for tier in (0, 1, 2):
        ordered.extend(_diverse_round_robin(tiers[tier], seed=seed + tier))
    return ordered


def build_queue(df: pd.DataFrame, targets: dict, seed: int, oversample: float) -> pd.DataFrame:
    picks = []
    for level, target in sorted(targets.items()):
        df_level = df[df['difficulty'] == level]
        ranked = rank_level_candidates(df_level, seed=seed + level * 1000)
        queue_size = min(len(ranked), max(target, int(round(target * oversample))))
        for rank, row in enumerate(ranked[:queue_size]):
            r = dict(row)
            r['rank_in_level'] = rank
            r['level_target'] = target
            r['in_primary_target'] = rank < target
            picks.append(r)
    return pd.DataFrame(picks)


df_queue_cfn = build_queue(df_cfn, cfn_targets, seed=RNG_SEED, oversample=OVERSAMPLE_FACTOR)
df_queue_tf = build_queue(df_tf, tf_targets, seed=RNG_SEED, oversample=OVERSAMPLE_FACTOR)

print(f"CFN queue: {len(df_queue_cfn)} candidates ({df_queue_cfn['in_primary_target'].sum()} primary + "
      f"{(~df_queue_cfn['in_primary_target']).sum()} spares)")
print(f"TF  queue: {len(df_queue_tf)} candidates ({df_queue_tf['in_primary_target'].sum()} primary + "
      f"{(~df_queue_tf['in_primary_target']).sum()} spares)")

CFN queue: 55 candidates (35 primary + 20 spares)
TF  queue: 55 candidates (35 primary + 20 spares)


## §5 Pre-check summary: what the primary selection looks like right now

In [7]:
def summarise(df_queue: pd.DataFrame, label: str):
    primary = df_queue[df_queue['in_primary_target']]
    print(f"=== {label}: primary selection ({len(primary)} scenarios) ===")
    print(primary.groupby('difficulty')['ls_status'].value_counts(dropna=False).unstack(fill_value=0))
    print(f"distinct diversity groups covered: {primary['diversity_key'].nunique()} / {len(primary)}")
    _needs_check = (primary['ls_status'] != True).sum()
    print(f"candidates needing a live LocalStack check before they can be trusted: {_needs_check}")
    print()
    return primary


primary_cfn = summarise(df_queue_cfn, 'CFN')
primary_tf = summarise(df_queue_tf, 'TF')

=== CFN: primary selection (35 scenarios) ===
ls_status   True
difficulty      
1              2
2              5
3              7
4              9
5             12
distinct diversity groups covered: 34 / 35
candidates needing a live LocalStack check before they can be trusted: 0

=== TF: primary selection (35 scenarios) ===
ls_status   True
difficulty      
1              2
2              5
3              7
4              9
5             12
distinct diversity groups covered: 35 / 35
candidates needing a live LocalStack check before they can be trusted: 0



## §6 Live LocalStack deploy check — CloudFormation (optional, gated by `RUN_LOCALSTACK_CHECK_CFN`)

In [8]:
def _cfn_localstack_client():
    import boto3
    return boto3.client(
        'cloudformation',
        region_name=AWS_REGION,
        endpoint_url=LOCALSTACK_ENDPOINT,
        aws_access_key_id='test',
        aws_secret_access_key='test',
        aws_session_token='test',
    )


def cfn_validate_template(content: str, ext: str) -> dict:
    suffix = ext if ext.startswith('.') else f'.{ext}'
    with tempfile.NamedTemporaryFile(suffix=suffix, mode='w', encoding='utf-8', delete=False) as f:
        f.write(content)
        tmp_path = f.name
    try:
        cmd = ['aws', '--endpoint-url', LOCALSTACK_ENDPOINT, 'cloudformation', 'validate-template',
               '--template-body', f'file://{tmp_path}', '--region', AWS_REGION]
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=60)
        if result.returncode == 0:
            return {'validate_pass': True, 'validate_error': ''}
        return {'validate_pass': False, 'validate_error': (result.stderr or result.stdout or '').strip()[:500]}
    except subprocess.TimeoutExpired:
        return {'validate_pass': None, 'validate_error': 'timeout'}
    finally:
        Path(tmp_path).unlink(missing_ok=True)


def _cfn_stack_failure_reason(cfn, stack_name: str) -> str:
    try:
        events = cfn.describe_stack_events(StackName=stack_name)['StackEvents']
    except Exception:
        return ''
    for ev in events:
        if str(ev.get('ResourceStatus', '')).endswith('FAILED'):
            reason = ev.get('ResourceStatusReason', '')
            if reason:
                return f"{ev.get('LogicalResourceId', '')}: {reason}"[:500]
    return ''


def cfn_deploy_stack(content: str) -> dict:
    import botocore
    cfn = _cfn_localstack_client()
    stack_name = f'{AWS_STACK_PREFIX}-{uuid.uuid4().hex[:10]}'
    try:
        cfn.create_stack(
            StackName=stack_name, TemplateBody=content,
            Capabilities=['CAPABILITY_IAM', 'CAPABILITY_NAMED_IAM', 'CAPABILITY_AUTO_EXPAND'],
        )
        waiter = cfn.get_waiter('stack_create_complete')
        waiter.wait(StackName=stack_name,
                    WaiterConfig={'Delay': 10, 'MaxAttempts': max(1, int(DEPLOY_TIMEOUT_MIN * 60 / 10))})
        status = cfn.describe_stacks(StackName=stack_name)['Stacks'][0].get('StackStatus', 'UNKNOWN')
        result = {'deploy_pass': True, 'deploy_status': status, 'deploy_error': '', 'deployed_stack_name': stack_name}
    except botocore.exceptions.WaiterError as e:
        try:
            desc = cfn.describe_stacks(StackName=stack_name)['Stacks'][0]
            status = desc.get('StackStatus', 'UNKNOWN')
            reason = desc.get('StackStatusReason', '') or _cfn_stack_failure_reason(cfn, stack_name)
            if not reason:
                reason = f'{status}: no StackStatusReason or FAILED resource event found'
        except Exception:
            status, reason = 'WAIT_FAILED', str(e)[:500]
        result = {'deploy_pass': False, 'deploy_status': status, 'deploy_error': reason,
                  'deployed_stack_name': stack_name}
    except Exception as e:
        result = {'deploy_pass': False, 'deploy_status': 'ERROR', 'deploy_error': str(e)[:500],
                  'deployed_stack_name': stack_name}
    finally:
        try:
            cfn.delete_stack(StackName=stack_name)
            cfn.get_waiter('stack_delete_complete').wait(
                StackName=stack_name,
                WaiterConfig={'Delay': 10, 'MaxAttempts': max(1, int(DEPLOY_TIMEOUT_MIN * 60 / 10))})
        except Exception:
            pass
    return result


def cfn_localstack_check_one(dest_file: str) -> dict:
    path = CFN_BASE_DIR / dest_file
    content = path.read_text(encoding='utf-8', errors='ignore')
    ext = path.suffix
    v = cfn_validate_template(content, ext)
    if not v['validate_pass']:
        return {**v, 'deploy_pass': False, 'deploy_status': 'VALIDATE_FAILED',
                'deploy_error': v['validate_error'], 'deployed_stack_name': ''}
    d = cfn_deploy_stack(content)
    return {**v, **d}

In [9]:
if FREEZE_ORIGINAL_SELECTION or not RUN_LOCALSTACK_CHECK_CFN:
    print('⏸️  Part A frozen or RUN_LOCALSTACK_CHECK_CFN is False -- skipping the live CFN check '
          '(the queue already has a prior-pass column to work from).')
else:
    try:
        _h = requests.get(f'{LOCALSTACK_ENDPOINT}/_localstack/health', timeout=5)
        print(f'LocalStack health: HTTP {_h.status_code}')
    except requests.exceptions.RequestException as e:
        raise RuntimeError(f'LocalStack not reachable at {LOCALSTACK_ENDPOINT} -- start it first ({e})')

    _to_check = df_queue_cfn[df_queue_cfn['ls_status'] != True].drop_duplicates('dest_file')
    print(f'Live-checking {len(_to_check)} CFN candidate(s) against LocalStack...')

    _results = []
    for i, (_, row) in enumerate(_to_check.iterrows(), 1):
        print(f"[{i}/{len(_to_check)}] {row['dest_file']}", end=' -> ')
        res = cfn_localstack_check_one(row['dest_file'])
        print('PASS' if res['deploy_pass'] else f"FAIL ({res['deploy_error'][:120]})")
        _results.append({'content_hash': row['fresh_content_hash'], **res})

    if _results:
        _df_new = pd.DataFrame(_results)
        _existing_cols = pd.read_csv(CFN_LOCALSTACK_CACHE, nrows=0).columns
        _df_new = _df_new.reindex(columns=_existing_cols)
        _df_new.to_csv(CFN_LOCALSTACK_CACHE, mode='a', header=False, index=False)
        print(f"\nAppended {len(_df_new)} fresh result(s) to {CFN_LOCALSTACK_CACHE}")

        _fresh_map = dict(zip(_df_new['content_hash'], _df_new['deploy_pass']))
        df_cfn.loc[df_cfn['fresh_content_hash'].isin(_fresh_map), 'ls_status'] = \
            df_cfn.loc[df_cfn['fresh_content_hash'].isin(_fresh_map), 'fresh_content_hash'].map(_fresh_map)
        df_queue_cfn['ls_status'] = df_queue_cfn['fresh_content_hash'].map(
            lambda h: _fresh_map.get(h, None)).combine_first(df_queue_cfn['ls_status'])
        print('Re-run §5 (and §8 below) to see the updated selection.')

⏸️  Part A frozen or RUN_LOCALSTACK_CHECK_CFN is False -- skipping the live CFN check (the queue already has a prior-pass column to work from).


## §7 Live LocalStack deploy check — Terraform (optional, gated by `RUN_LOCALSTACK_CHECK_TF`)

In [10]:
LOCALSTACK_RUN_ENV = os.environ.copy()
LOCALSTACK_RUN_ENV.update({
    'AWS_ACCESS_KEY_ID': 'test',
    'AWS_SECRET_ACCESS_KEY': 'test',
    'AWS_DEFAULT_REGION': AWS_REGION,
    'TF_PLUGIN_CACHE_DIR': str(TF_PLUGIN_CACHE_DIR),
    'TF_PLUGIN_CACHE_MAY_BREAK_DEPENDENCY_LOCK_FILE': 'true',
    'PATH': f"/opt/homebrew/bin:{LOCALSTACK_RUN_ENV.get('PATH', '')}",
})
LOCALSTACK_RUN_ENV.pop('AWS_PROFILE', None)


def tf_deploy_localstack(folder_path: str):
    folder_path = Path(folder_path)
    deploy_passed = False
    init_log, apply_log, err_log = '', '', ''
    with tempfile.TemporaryDirectory() as tmpdir:
        tmp_path = Path(tmpdir)
        for f in folder_path.glob('*.tf'):
            shutil.copy2(f, tmp_path / f.name)
        try:
            init_res = subprocess.run(['tflocal', 'init', '-backend=false', '-input=false', '-no-color'],
                                       cwd=tmpdir, capture_output=True, text=True, timeout=5 * 60,
                                       env=LOCALSTACK_RUN_ENV)
            init_log = init_res.stdout + '\n' + init_res.stderr
            if init_res.returncode == 0:
                apply_res = subprocess.run(['tflocal', 'apply', '-auto-approve', '-input=false', '-no-color'],
                                            cwd=tmpdir, capture_output=True, text=True, timeout=10 * 60,
                                            env=LOCALSTACK_RUN_ENV)
                apply_log = apply_res.stdout + '\n' + apply_res.stderr
                deploy_passed = apply_res.returncode == 0
                subprocess.run(['tflocal', 'destroy', '-auto-approve', '-input=false', '-no-color'],
                                cwd=tmpdir, capture_output=True, text=True, timeout=5 * 60, env=LOCALSTACK_RUN_ENV)
        except subprocess.TimeoutExpired as e:
            cmd_name = e.cmd[1] if len(e.cmd) > 1 else 'unknown'
            err_log = f"TIMEOUT: '{cmd_name}' exceeded its time limit."
        except Exception as e:
            err_log = f'SYSTEM ERROR: {e}'
    combined_log = f'--- INIT ---\n{init_log}\n\n--- APPLY ---\n{apply_log}\n\n--- ERRORS ---\n{err_log}'.strip()
    if len(combined_log) > 4000:
        combined_log = combined_log[:1000] + '\n[...]\n' + combined_log[-3000:]
    return deploy_passed, combined_log

In [11]:
if FREEZE_ORIGINAL_SELECTION or not RUN_LOCALSTACK_CHECK_TF:
    print('⏸️  Part A frozen or RUN_LOCALSTACK_CHECK_TF is False -- skipping the live TF check '
          '(the queue already has a (soft) prior-pass column to work from).')
else:
    try:
        _h = requests.get(f'{LOCALSTACK_ENDPOINT}/_localstack/health', timeout=5)
        print(f'LocalStack health: HTTP {_h.status_code}')
    except requests.exceptions.RequestException as e:
        raise RuntimeError(f'LocalStack not reachable at {LOCALSTACK_ENDPOINT} -- start it first ({e})')

    if RECHECK_TF_CONFIRMED:
        _to_check = df_queue_tf.drop_duplicates('scenario_id')
    else:
        _to_check = df_queue_tf[df_queue_tf['ls_status'] != True].drop_duplicates('scenario_id')
    print(f'Live-checking {len(_to_check)} TF candidate(s) against LocalStack '
          f"(RECHECK_TF_CONFIRMED={RECHECK_TF_CONFIRMED})...")

    _results = []
    for i, (_, row) in enumerate(_to_check.iterrows(), 1):
        print(f"[{i}/{len(_to_check)}] {row['scenario_id']}", end=' -> ')
        passed, log = tf_deploy_localstack(row['folder_path'])
        print('PASS' if passed else 'FAIL')
        _results.append({
            'scenario_id': row['scenario_id'], 'source': row.get('source', ''),
            'folder_path': row['folder_path'], 'deploy_passed': passed, 'deploy_log': log,
            'loc': row.get('loc'), 'tokens': row.get('tokens'),
            'n_resources': row.get('n_resources'), 'n_params': row.get('n_params'),
            'difficulty': row.get('difficulty'),
        })

    if _results:
        _df_new = pd.DataFrame(_results)
        _existing_cols = pd.read_csv(TF_LOCALSTACK_CACHE, nrows=0).columns
        _df_new = _df_new.reindex(columns=_existing_cols)
        _df_new.to_csv(TF_LOCALSTACK_CACHE, mode='a', header=False, index=False)
        print(f"\nAppended {len(_df_new)} fresh result(s) to {TF_LOCALSTACK_CACHE}")

        _fresh_map = dict(zip(_df_new['scenario_id'], _df_new['deploy_passed']))
        df_tf.loc[df_tf['scenario_id'].isin(_fresh_map), 'ls_status'] = \
            df_tf.loc[df_tf['scenario_id'].isin(_fresh_map), 'scenario_id'].map(_fresh_map)
        df_queue_tf['ls_status'] = df_queue_tf['scenario_id'].map(
            lambda s: _fresh_map.get(s, None)).combine_first(df_queue_tf['ls_status'])
        print('Re-run §5 (and §8 below) to see the updated selection.')

⏸️  Part A frozen or RUN_LOCALSTACK_CHECK_TF is False -- skipping the live TF check (the queue already has a (soft) prior-pass column to work from).


## §8 Final assembly

In [12]:
def assemble_final(df_queue: pd.DataFrame, targets: dict, label: str) -> pd.DataFrame:
    final_rows = []
    shortfalls = {}
    for level, target in sorted(targets.items()):
        level_queue = df_queue[df_queue['difficulty'] == level].sort_values('rank_in_level')
        confirmed = level_queue[level_queue['ls_status'] == True]
        chosen = confirmed.head(target)
        final_rows.append(chosen)
        if len(chosen) < target:
            shortfalls[level] = target - len(chosen)
    df_final = pd.concat(final_rows, ignore_index=True) if final_rows else pd.DataFrame()
    print(f"=== {label}: final selection = {len(df_final)} / {sum(targets.values())} "
          f"({df_final['diversity_key'].nunique() if len(df_final) else 0} distinct diversity groups) ===")
    if shortfalls:
        print(f"⚠️ shortfall by level: {shortfalls} -- raise OVERSAMPLE_FACTOR / rerun the live check to close this")
    else:
        print('✅ every level fully covered by confirmed LocalStack-pass scenarios.')
    return df_final


df_final_cfn = assemble_final(df_queue_cfn, cfn_targets, 'CFN')
print()
df_final_tf = assemble_final(df_queue_tf, tf_targets, 'TF')

=== CFN: final selection = 35 / 35 (34 distinct diversity groups) ===
✅ every level fully covered by confirmed LocalStack-pass scenarios.

=== TF: final selection = 35 / 35 (35 distinct diversity groups) ===
✅ every level fully covered by confirmed LocalStack-pass scenarios.


## §9 Selection report

In [13]:
def level_report(df_final: pd.DataFrame, label: str):
    print(f"--- {label} ---")
    print(df_final['difficulty'].value_counts().sort_index())
    print(f"distinct aws_services groups: {df_final['diversity_key'].nunique()} / {len(df_final)}")
    print()


level_report(df_final_cfn, 'CFN')
level_report(df_final_tf, 'TF')

print(f"TOTAL selected: {len(df_final_cfn) + len(df_final_tf)} / {TOTAL_SCENARIOS}")

--- CFN ---
difficulty
1     2
2     5
3     7
4     9
5    12
Name: count, dtype: int64
distinct aws_services groups: 34 / 35

--- TF ---
difficulty
1     2
2     5
3     7
4     9
5    12
Name: count, dtype: int64
distinct aws_services groups: 35 / 35

TOTAL selected: 70 / 70


## §10 Export

In [14]:
if FREEZE_ORIGINAL_SELECTION:
    print('🔒 FREEZE_ORIGINAL_SELECTION is True -- Part A export skipped; the original subset is '
          'preserved (see Part B, §11-§17).')
else:
    df_selected = pd.concat([
        df_final_cfn.assign(lang='cfn')[['lang', 'row_number', 'difficulty', 'path_key',
                                          'diversity_key', 'ls_status', 'user_prompt']],
        df_final_tf.assign(lang='tf')[['lang', 'row_number', 'difficulty', 'path_key',
                                        'diversity_key', 'ls_status', 'user_prompt']],
    ], ignore_index=True)
    df_selected = df_selected.rename(columns={'path_key': 'ground_truth_path', 'user_prompt': 'prompt'})
    df_selected.to_csv(SELECTION_CSV, index=False)
    print(f"Wrote {len(df_selected)} rows -> {SELECTION_CSV}")

    _cfn_out = df_final_cfn[['row_number', 'path_key', 'user_prompt', 'difficulty']].rename(
        columns={'path_key': 'ground_truth_path', 'user_prompt': 'prompt'}).sort_values('row_number')
    _tf_out = df_final_tf[['row_number', 'path_key', 'user_prompt', 'difficulty']].rename(
        columns={'path_key': 'ground_truth_path', 'user_prompt': 'prompt'}).sort_values('row_number')

    for _df, _local, _iacgod, _label in [
        (_cfn_out, CFN_ABLATION_EVAL, CFN_ABLATION_EVAL_IACGOD, 'CFN'),
        (_tf_out, TF_ABLATION_EVAL, TF_ABLATION_EVAL_IACGOD, 'TF'),
    ]:
        assert _df['row_number'].duplicated().sum() == 0, f'{_label}: duplicate row_number in final selection'
        _df.to_csv(_local, index=False)
        _df.to_csv(_iacgod, index=False)
        print(f"{_label}: {len(_df)} rows -> {_local}")
        print(f"{_label}: {len(_df)} rows -> {_iacgod}")

🔒 FREEZE_ORIGINAL_SELECTION is True -- Part A export skipped; the original subset is preserved (see Part B, §11-§17).


## Next steps

# Part B — Extending the ablation benchmark to 50 scenarios per language (2026-09-27)

## §11 Part B configuration

In [15]:
IACGOD_DIR = (DATA_ANALYSIS_DIR / '..').resolve()
RESULT_DIR = DATA_ANALYSIS_DIR / 'result' / 'iacgod'

EXTENDED_TARGET_PER_LANG = 50
EXT_SPARES_PER_LEVEL     = 3
EXT_RUN_LOCALSTACK_CHECK = True
EXT_EXPORT               = True

FULL_RUNS = {
    'cfn': {
        'dsv4f':    IACGOD_DIR / 'benchmark_runs/cloudformation_20260917_170146_DSV4F_Full/CFNEvalRealAWS_DeepseekV4Flash_sec_runs.csv',
        'glm53':    RESULT_DIR / 'CFNEvalRealAWS_GLM53Flash_sec_runs.csv',
        'opencode': RESULT_DIR / 'CFNEvalRealAWS_Opencode_DeepseekV4Flash_sec_runs.csv',
    },
    'tf': {
        'dsv4f':    IACGOD_DIR / 'benchmark_runs/terraform_20260920_135237_DSV4_Full/TFEvalRealAWS_DeepseekV4Flash_sec_runs.csv',
    },
}
RUNS_DIR   = IACGOD_DIR / 'runs'
RETRY_LIST = RESULT_DIR / 'Retry_Candidates_20260927.csv'
RETRY_LIST_EVALUATION = {
    ('cfn', 'dsv4f'):    'CFN MAS DeepSeek (live AWS)',
    ('cfn', 'glm53'):    'CFN MAS GLM-5.3 (live AWS)',
    ('cfn', 'opencode'): 'CFN OpenCode DeepSeek (live AWS)',
    ('tf', 'dsv4f'):     'TF MAS DeepSeek (live AWS)',
}
TF_PLAN_CACHE = TF_DATASET_DIR / 'tf_plan_resource_cache.csv'

HARD_ITER     = 6
MODERATE_ITER = 4
NEAR_VACUOUS_TF_RESOURCES = 2

DIFFICULTY_CSV        = OUT_DIR / 'full_benchmark_difficulty.csv'
EXT_CHECK_LOG         = OUT_DIR / 'extension_localstack_checks.csv'
SELECTION_V2_CSV      = OUT_DIR / 'ablation_benchmark_selection_v2.csv'
BASE_SNAPSHOT = {l: OUT_DIR / f'{l}_eval_benchmark_ablation_v1_35.csv' for l in ('cfn', 'tf')}
ADDITIONS_CSV = {l: OUT_DIR / f'{l}_eval_benchmark_ablation_additions.csv' for l in ('cfn', 'tf')}
ADDITIONS_CSV_IACGOD = {l: IACGOD_DATA_DIR / f'{l}_eval_benchmark_ablation_additions.csv' for l in ('cfn', 'tf')}
ABLATION_EVAL = {'cfn': (CFN_ABLATION_EVAL, CFN_ABLATION_EVAL_IACGOD),
                 'tf':  (TF_ABLATION_EVAL, TF_ABLATION_EVAL_IACGOD)}

print(f'Extended target: {EXTENDED_TARGET_PER_LANG} per language | '
      f'LocalStack check: {EXT_RUN_LOCALSTACK_CHECK} | export: {EXT_EXPORT}')

Extended target: 50 per language | LocalStack check: True | export: True


## §12 Difficulty evidence from the full-benchmark runs

In [16]:
import csv
import glob

csv.field_size_limit(2**31 - 1)

ENV_ERROR_RE = re.compile(
    r'already exists|AlreadyExists|BucketAlreadyOwnedByYou|ServiceLimitExceeded|LimitExceeded|'
    r'Maximum allowed resources exceeded|quota|InternalFailure|Failed to install provider|'
    r'Failed to query available provider|Could not connect|connection reset|i/o timeout|'
    r'TLS handshake|no space left', re.I)
STAGE_CLASS = {'yaml': 'syntax_lint', 'cfn-lint': 'syntax_lint', 'tflint': 'syntax_lint',
               'terraform-validate': 'syntax_lint', 'trivy': 'security'}
EVAL_CSV = {'cfn': CFN_EVAL_CSV, 'tf': TF_EVAL_CSV}


def _load_run(path):
    d = pd.read_csv(path, engine='python', usecols=lambda c: c in (
        'row_number', 'ground_truth_path', 'run_id', 'final_validation_passed', 'iterations_used'))
    d = d.dropna(subset=['row_number']).drop_duplicates('row_number', keep='last')
    d['row_number'] = d['row_number'].astype(float).astype(int)
    d['passed'] = d['final_validation_passed'].astype(str).str.lower().eq('true')
    return d


def _same_benchmark(d, lang, label):
    cur = pd.read_csv(EVAL_CSV[lang]).set_index('row_number')['ground_truth_path'].astype(str).str.strip()
    ok = [cur.get(r, None) == str(g).strip() for r, g in zip(d['row_number'], d['ground_truth_path'])]
    if not all(ok):
        print(f'  ⚠️ {lang}/{label}: {len(ok) - sum(ok)} row(s) ran a different ground truth '
              f'under the same row_number -- ignored as evidence')
    return d[ok]


def _retry_listed(lang, label):
    if not RETRY_LIST.exists():
        return set()
    rt = pd.read_csv(RETRY_LIST)
    name = RETRY_LIST_EVALUATION.get((lang, label))
    rt = rt[(rt['evaluation'] == name) & rt['action'].astype(str).str.startswith('retry (')]
    return set(rt['row_number'].astype(int))


def _index_run_dirs(run_ids):
    run_ids, found = set(map(str, run_ids)), {}
    for dp, _, fs in os.walk(RUNS_DIR):
        name = os.path.basename(dp)
        if name in run_ids and any(f.startswith('iteration_') for f in fs):
            found.setdefault(name, dp)
    return found


def _classify_iteration(j):
    for v in j.get('validation_results') or []:
        if not v.get('passed'):
            msg = ' | '.join(map(str, v.get('errors') or []))[:2000] or str(v.get('raw_output'))[:1000]
            stage = v.get('stage')
            if stage == 'terraform-validate' and ENV_ERROR_RE.search(msg):
                return 'env', msg
            return STAGE_CLASS.get(stage, 'syntax_lint'), msg
    if j.get('validation_passed'):
        return 'pass', ''
    dep = j.get('deploy_validation_result') or {}
    msg = (str(dep.get('error_message') or '') + ' ' +
           ' '.join(str(x.get('status_reason')) for x in dep.get('failed_resources') or []))[:2000]
    return ('env' if ENV_ERROR_RE.search(msg) else 'deploy'), msg


def primary_run_evidence(lang, path):
    d = _same_benchmark(_load_run(path), lang, 'dsv4f')
    dirs = _index_run_dirs(d['run_id'])
    recs = []
    for _, r in d.iterrows():
        counts = {'syntax_lint': 0, 'security': 0, 'deploy': 0, 'env': 0, 'pass': 0}
        last_msg = ''
        for p in sorted(glob.glob(f"{dirs.get(str(r['run_id']), '/nonexistent')}/iteration_*.json")):
            cls, msg = _classify_iteration(json.load(open(p)))
            counts[cls] += 1
            last_msg = msg or last_msg
        n = sum(counts.values())
        if n == 0:
            n = int(r['iterations_used']) if pd.notna(r['iterations_used']) else 0
        recs.append({'row_number': r['row_number'], 'dsv4f_passed': bool(r['passed']),
                     'dsv4f_iterations': n, 'dsv4f_env_iterations': counts['env'],
                     'dsv4f_genuine_iterations': n - counts['env'],
                     'iters_syntax_lint': counts['syntax_lint'], 'iters_security': counts['security'],
                     'iters_deploy': counts['deploy'], 'dsv4f_last_error': last_msg[:300]})
    out = pd.DataFrame(recs)
    out['env_retry_listed'] = out['row_number'].isin(_retry_listed(lang, 'dsv4f'))
    missing = len(d) - len(dirs)
    if missing:
        print(f'  ⚠️ {lang}: {missing} dsv4f run(s) have no iteration files -- CSV iteration counts used')
    return out


def _tier(r):
    if not r['dsv4f_passed']:
        return 'A_failed'
    if r['dsv4f_genuine_iterations'] >= HARD_ITER:
        return 'B_hard_pass'
    if r['dsv4f_genuine_iterations'] >= MODERATE_ITER:
        return 'C_moderate'
    return 'D_easy'


def build_evidence(lang):
    ev = primary_run_evidence(lang, FULL_RUNS[lang]['dsv4f'])
    ev['tier'] = ev.apply(_tier, axis=1)
    ev['hard_dsv4f'] = ev['tier'].isin(['A_failed', 'B_hard_pass'])
    hard_cols = ['hard_dsv4f']
    for label, path in FULL_RUNS[lang].items():
        if label == 'dsv4f' or not Path(path).exists():
            continue
        d = _same_benchmark(_load_run(path), lang, label)
        env_rows = _retry_listed(lang, label)
        d[f'hard_{label}'] = [None if r in env_rows else bool((not p) or it >= HARD_ITER)
                              for r, p, it in zip(d['row_number'], d['passed'], d['iterations_used'])]
        d = d.rename(columns={'iterations_used': f'{label}_iterations', 'passed': f'{label}_passed'})
        ev = ev.merge(d[['row_number', f'{label}_iterations', f'{label}_passed', f'hard_{label}']],
                      on='row_number', how='left')
        hard_cols.append(f'hard_{label}')
    _is_bool = lambda v: isinstance(v, (bool, np.bool_))
    ev['n_runs_hard'] = ev[hard_cols].apply(lambda r: sum(_is_bool(v) and bool(v) for v in r), axis=1)
    ev['n_runs_evidence'] = ev[hard_cols].apply(lambda r: sum(_is_bool(v) for v in r), axis=1)
    if lang == 'tf' and TF_PLAN_CACHE.exists():
        plan = pd.read_csv(TF_PLAN_CACHE, usecols=['scenario_id', 'n_plan_aws']).drop_duplicates('scenario_id', keep='last')
        ids = df_tf[['row_number', 'scenario_id']]
        ev = ev.merge(ids, on='row_number', how='left').merge(plan, on='scenario_id', how='left').drop(columns='scenario_id')
    ev['lang'] = lang
    return ev


evidence = {lang: build_evidence(lang) for lang in ('cfn', 'tf')}
pd.concat(evidence.values(), ignore_index=True).to_csv(DIFFICULTY_CSV, index=False)
print(f'Wrote {DIFFICULTY_CSV}')
for lang, ev in evidence.items():
    lvl = (df_cfn if lang == 'cfn' else df_tf)[['row_number', 'difficulty']]
    x = ev.merge(lvl, on='row_number')
    print(f'\n=== {lang.upper()}: tier x difficulty level (primary run, environment iterations removed) ===')
    print(pd.crosstab(x['tier'], x['difficulty'], margins=True))
    print('genuine failing iterations by stage:',
          x[['iters_syntax_lint', 'iters_security', 'iters_deploy']].sum().to_dict(),
          '| environment iterations removed:', int(x['dsv4f_env_iterations'].sum()))
    if 'hard_glm53' in x:
        print('runs agreeing a scenario is hard (n_runs_hard):', x['n_runs_hard'].value_counts().sort_index().to_dict())

Wrote /Users/iksena/Documents/research/data_analysis/ablation_selection/full_benchmark_difficulty.csv

=== CFN: tier x difficulty level (primary run, environment iterations removed) ===
difficulty    1   2   3   4   5  All
tier                                
A_failed      3   2   2   3   1   11
B_hard_pass   2   5  10   9  11   37
C_moderate    1   5  11  11  14   42
D_easy       44  38  27  27  24  160
All          50  50  50  50  50  250
genuine failing iterations by stage: {'iters_syntax_lint': 253, 'iters_security': 95, 'iters_deploy': 343} | environment iterations removed: 84
runs agreeing a scenario is hard (n_runs_hard): {0: 161, 1: 51, 2: 28, 3: 10}

=== TF: tier x difficulty level (primary run, environment iterations removed) ===
difficulty    1   2   3   4   5  All
tier                                
A_failed      0   0   0   1   3    4
B_hard_pass   2   2   2   5   6   17
C_moderate    1   1   3   5  16   26
D_easy       47  47  45  39  25  203
All          50  50  50  50 

## §13 The frozen original subset and how many scenarios to add

In [17]:
base = {}
for lang in ('cfn', 'tf'):
    if not BASE_SNAPSHOT[lang].exists():
        shutil.copy2(ABLATION_EVAL[lang][0], BASE_SNAPSHOT[lang])
        print(f'Snapshot of the original subset -> {BASE_SNAPSHOT[lang]}')
    base[lang] = pd.read_csv(BASE_SNAPSHOT[lang])

    full = (df_cfn if lang == 'cfn' else df_tf)[['row_number', 'path_key', 'user_prompt', 'difficulty']]
    chk = base[lang].merge(full, on='row_number', how='left', suffixes=('', '_now'))
    drift = chk[(chk['path_key'] != chk['ground_truth_path']) | (chk['user_prompt'] != chk['prompt'])
                | (chk['difficulty_now'] != chk['difficulty'])]
    assert drift.empty, (f'{lang}: original rows {drift.row_number.tolist()} no longer match the full '
                         'benchmark -- resolve before extending')

additions_needed = {}
for lang in ('cfn', 'tf'):
    target = allocate_targets(EXTENDED_TARGET_PER_LANG, LEVEL_WEIGHTS)
    have = base[lang]['difficulty'].value_counts().to_dict()
    additions_needed[lang] = {l: max(0, target[l] - have.get(l, 0)) for l in sorted(target)}
    print(f'{lang.upper()}: original {len(base[lang])} {dict(sorted(have.items()))} -> target '
          f'{target} -> add {additions_needed[lang]} ({sum(additions_needed[lang].values())} total)')

CFN: original 35 {1: 2, 2: 5, 3: 7, 4: 9, 5: 12} -> target {1: 3, 2: 7, 3: 10, 4: 13, 5: 17} -> add {1: 1, 2: 2, 3: 3, 4: 4, 5: 5} (15 total)
TF: original 35 {1: 2, 2: 5, 3: 7, 4: 9, 5: 12} -> target {1: 3, 2: 7, 3: 10, 4: 13, 5: 17} -> add {1: 1, 2: 2, 3: 3, 4: 4, 5: 5} (15 total)


## §14 Manual pick / priority cell (edit this cell)

In [18]:
MANUAL_PICKS = {
    'cfn': {
        5:   'L1 Step Functions -> custom EventBridge bus; 10 genuine iterations',
        47:  'L2 DevOps Agent Space + roles; hard in 2 runs (DeepSeek 7, OpenCode 15)',
        49:  'L2 Lambda blue/green via CodeDeploy; new service group',
        77:  'L3 App Runner + ECR; failed in the full run (15 iterations), GLM 11',
        103: 'L3 SageMaker notebook (RAPIDS); DeepSeek 8, GLM 15',
        85:  'L3 scheduled Organizations account-closure workflow; DeepSeek 7, GLM 11',
        119: 'L4 ECS Fargate + Cloud Map discovery; failed in DeepSeek and GLM runs',
        150: 'L4 FIS chaos-experiment template; hard in all 3 runs (LocalStack status unknown)',
        131: 'L4 Bedrock AgentCore Gateway + Cognito; DeepSeek 8, GLM 13',
        138: 'L4 MediaPackage channel + HLS/DASH endpoints; DeepSeek 6, GLM 15',
        176: 'L5 dual-stack IPv4/IPv6 VPC; 14 genuine iterations',
        161: 'L5 landing-zone hub VPC; DeepSeek 8, OpenCode 11',
        262: 'L5 AppSync GraphQL API + DynamoDB with JS resolvers; 6 genuine iterations',
        252: 'L5 reusable DynamoDB table stack with alarms/backup; DeepSeek 6, GLM 9 (LocalStack unknown)',
        166: 'L5 shared EKS IAM roles + optional GuardDuty; OpenCode 20 iterations',
    },
    'tf': {
        319: 'L1 AWS Config managed rule; 9 genuine iterations',
        339: 'L2 VPC with public/private subnets and NAT; 10 genuine iterations',
        56:  'L2 Aurora DSQL cluster (optional peering); 7 genuine iterations',
        286: 'L3 API Gateway v2 WebSocket -> SQS; 18 planned resources',
        122: 'L3 internal ALB with path-based routing (self-signed TLS); LocalStack unknown',
        114: 'L3 feature-complete S3 bucket module',
        311: 'L4 S3 Multi-Region Access Point + replicated buckets; 11 genuine iterations',
        148: 'L4 API Gateway -> DynamoDB direct integration; 6 genuine iterations',
        156: 'L4 n8n on ECS Fargate + EFS (42 planned resources)',
        199: 'L4 static site: private S3 + CloudFront + ACM + Route 53 (LocalStack unknown)',
        164: 'L5 self-contained multi-tier network; 10 genuine iterations',
        190: 'L5 dual-stack three-tier network; 27 planned resources',
        195: 'L5 CloudTrail logging architecture (KMS, S3, CloudWatch)',
        186: 'L5 Transfer Family SFTP server with user automation',
        168: 'L5 enterprise S3 bucket architecture (replication, logging)',
    },
}

MANUAL_EXCLUDE = {
    'cfn': {
        2:   'failed on the environment (AWS Backup framework already exists in the account); on the retry list',
        20:  'Access Analyzer: fails in all 3 runs at 15 iterations, likely account-level',
        106: 'failed after 1 genuine iteration: environment failure, on the retry list',
        263: 'Config rule-list template (5 identical rule resources); on the retry list',
        58:  'AWS App Mesh -- service retires 2026-09-30',
        174: 'Elastic Beanstalk solution stack pinned to a point release that AWS retires regularly',
    },
    'tf': {
        249: 'failed on the environment (full disk / provider install); on the retry list',
    },
}


def _diverse_order(rows, covered_keys):
    seen, first, later = set(covered_keys), [], []
    for r in rows:
        (later if r['diversity_key'] in seen else first).append(r)
        seen.add(r['diversity_key'])
    return first + later


def build_extension_queue(lang):
    df = (df_cfn if lang == 'cfn' else df_tf).copy()
    df['row_number'] = df['row_number'].astype(float).astype(int)
    df = df.merge(evidence[lang], on='row_number', how='left')
    base_rows = set(base[lang]['row_number'])
    picks, excl = MANUAL_PICKS[lang], MANUAL_EXCLUDE[lang]
    for r, why in picks.items():
        if r in base_rows:
            print(f'  ⚠️ {lang} pick {r} is already in the original subset -- ignored')
        elif r in excl:
            print(f'  ⚠️ {lang} pick {r} is also in MANUAL_EXCLUDE -- exclusion wins')
        elif r not in set(df['row_number']):
            print(f'  ⚠️ {lang} pick {r} is not in the full benchmark -- ignored')
    pool = df[~df['row_number'].isin(base_rows) & ~df['row_number'].isin(excl)]
    covered = set(df.loc[df['row_number'].isin(base_rows), 'diversity_key'])
    tier_rank = {'A_failed': 0, 'B_hard_pass': 1, 'C_moderate': 2, 'D_easy': 3}
    queue = []
    for level, need in additions_needed[lang].items():
        lvl = pool[pool['difficulty'] == level]
        manual = [lvl[lvl['row_number'] == r].iloc[0].to_dict() for r in picks if r in set(lvl['row_number'])]
        for m in manual:
            m.update(role='manual', reason=picks[m['row_number']])
        auto = lvl[~lvl['row_number'].isin(picks)].copy()
        auto['_k_ls_fail'] = auto['ls_status'].map(lambda v: v is False)
        auto['_k_env'] = auto['env_retry_listed'].fillna(False).astype(bool)
        auto['_k_vacuous'] = (auto['n_plan_aws'] <= NEAR_VACUOUS_TF_RESOURCES) if 'n_plan_aws' in auto else False
        auto['_k_tier'] = auto['tier'].map(tier_rank).fillna(3)
        auto = auto.sort_values(['_k_ls_fail', '_k_env', '_k_vacuous', '_k_tier', 'n_runs_hard',
                                 'dsv4f_genuine_iterations', 'row_number'],
                                ascending=[True, True, True, True, False, False, True])
        auto_rows = _diverse_order([r for _, r in auto.iterrows()],
                                   covered | {m['diversity_key'] for m in manual})
        n_auto = max(0, need - len(manual)) + EXT_SPARES_PER_LEVEL
        for a in auto_rows[:n_auto]:
            a = dict(a)
            a.update(role='auto', reason=f"auto: {a['tier']}, {int(a['dsv4f_genuine_iterations'])} genuine iterations, "
                                         f"hard in {int(a['n_runs_hard'])}/{int(a['n_runs_evidence'])} runs")
            manual.append(a)
        for rank, r in enumerate(manual):
            r.update(queue_rank=rank, level_need=need)
            queue.append(r)
    q = pd.DataFrame(queue)
    q['new_service_group'] = ~q['diversity_key'].isin(covered)
    return q


ext_queue = {lang: build_extension_queue(lang) for lang in ('cfn', 'tf')}
_show = ['difficulty', 'queue_rank', 'row_number', 'role', 'tier', 'dsv4f_genuine_iterations',
         'n_runs_hard', 'ls_status', 'new_service_group', 'reason']
for lang, q in ext_queue.items():
    print(f'\n=== {lang.upper()} extension queue (add {sum(additions_needed[lang].values())}; '
          f'{len(q)} queued incl. spares) ===')
    with pd.option_context('display.width', 250, 'display.max_colwidth', 90, 'display.max_rows', 200):
        print(q[[c for c in _show if c in q]].to_string(index=False))


=== CFN extension queue (add 15; 30 queued incl. spares) ===
 difficulty  queue_rank  row_number   role        tier  dsv4f_genuine_iterations  n_runs_hard ls_status  new_service_group                                                                                      reason
          1           0           5 manual B_hard_pass                        10            1      True               True                          L1 Step Functions -> custom EventBridge bus; 10 genuine iterations
          1           1          34   auto B_hard_pass                         7            1      True               True                                   auto: B_hard_pass, 7 genuine iterations, hard in 1/3 runs
          1           2         210   auto  C_moderate                         4            1      None               True                                    auto: C_moderate, 4 genuine iterations, hard in 1/3 runs
          1           3           3   auto      D_easy                        

## §15 LocalStack check of the extension queue

In [19]:
def _tf_fingerprint(folder_path):
    h = hashlib.sha256()
    for f in sorted(Path(folder_path).glob('*.tf')):
        h.update(f.name.encode()); h.update(f.read_bytes())
    return h.hexdigest()


def _fingerprint(lang, row):
    return row['fresh_content_hash'] if lang == 'cfn' else _tf_fingerprint(row['folder_path'])


_log_cols = ['lang', 'row_number', 'fingerprint', 'deploy_pass', 'deploy_error', 'checked_at']
ext_log = pd.read_csv(EXT_CHECK_LOG) if EXT_CHECK_LOG.exists() else pd.DataFrame(columns=_log_cols)
_known = {(l, int(r), fp): p for l, r, fp, p in zip(ext_log['lang'], ext_log['row_number'],
                                                   ext_log['fingerprint'], ext_log['deploy_pass'])}


def _append(path, rows):
    cols = pd.read_csv(path, nrows=0).columns
    pd.DataFrame(rows).reindex(columns=cols).to_csv(path, mode='a', header=False, index=False)


def _check_live(lang, row):
    if lang == 'cfn':
        res = cfn_localstack_check_one(row['dest_file'])
        _append(CFN_LOCALSTACK_CACHE, [{'content_hash': row['fresh_content_hash'], **res}])
        return bool(res['deploy_pass']), str(res.get('deploy_error') or '')[:300]
    passed, log = tf_deploy_localstack(row['folder_path'])
    _append(TF_LOCALSTACK_CACHE, [{
        'scenario_id': row['scenario_id'], 'source': row.get('source', ''), 'folder_path': row['folder_path'],
        'deploy_passed': passed, 'deploy_log': log, 'loc': row.get('loc'), 'tokens': row.get('tokens'),
        'n_resources': row.get('n_resources'), 'n_params': row.get('n_params'), 'difficulty': row.get('difficulty')}])
    err = '' if passed else (re.findall(r'Error: [^\n]{0,200}', log) or [log[-300:]])[-1]
    return bool(passed), err


if EXT_RUN_LOCALSTACK_CHECK:
    try:
        requests.get(f'{LOCALSTACK_ENDPOINT}/_localstack/health', timeout=5).raise_for_status()
    except requests.exceptions.RequestException as e:
        raise RuntimeError(f'LocalStack not reachable at {LOCALSTACK_ENDPOINT} -- start it first ({e})')

for lang, q in ext_queue.items():
    print(f'\n=== {lang.upper()} ===')
    q['ext_status'], q['ext_source'], q['ext_error'], q['accepted'] = None, '', '', False
    for level, need in additions_needed[lang].items():
        accepted = 0
        for i in q.index[q['difficulty'] == level]:
            if accepted >= need:
                q.at[i, 'ext_source'] = 'not needed'
                continue
            row = q.loc[i]
            fp = _fingerprint(lang, row)
            if lang == 'cfn' and row['ls_status'] in (True, False):
                status, src, err = row['ls_status'], 'cache (content hash)', ''
            elif (lang, int(row['row_number']), fp) in _known:
                status, src, err = bool(_known[(lang, int(row['row_number']), fp)]), 'extension log', ''
            elif EXT_RUN_LOCALSTACK_CHECK:
                print(f"  L{level} row {row['row_number']} ({row['role']}) deploying...", end=' ', flush=True)
                t0 = time.time()
                status, err = _check_live(lang, row)
                src = f'live ({time.time() - t0:.0f}s)'
                print('PASS' if status else f'FAIL {err[:150]}')
                _known[(lang, int(row['row_number']), fp)] = status
                pd.DataFrame([{'lang': lang, 'row_number': int(row['row_number']), 'fingerprint': fp,
                               'deploy_pass': status, 'deploy_error': err,
                               'checked_at': pd.Timestamp.now().isoformat(timespec='seconds')}]
                             ).to_csv(EXT_CHECK_LOG, mode='a', header=not EXT_CHECK_LOG.exists(), index=False)
            else:
                status, src, err = None, 'untested (check disabled)', ''
            q.at[i, 'ext_status'], q.at[i, 'ext_source'], q.at[i, 'ext_error'] = status, src, err
            if status is True:
                q.at[i, 'accepted'] = True
                accepted += 1
        flag = '✅' if accepted >= need else f'⚠️ short by {need - accepted}'
        print(f'  L{level}: {accepted}/{need} accepted {flag}')
    failed_picks = q[(q['role'] == 'manual') & (q['ext_status'] == False)]
    if len(failed_picks):
        print('  manual picks that failed on LocalStack (replaced by the next candidate):')
        for _, r in failed_picks.iterrows():
            print(f"    L{r['difficulty']} row {r['row_number']}: {str(r['ext_error'])[:160]}")


=== CFN ===
  L1: 1/1 accepted ✅
  L2: 2/2 accepted ✅
  L3: 3/3 accepted ✅
  L4: 4/4 accepted ✅
  L5: 5/5 accepted ✅
  manual picks that failed on LocalStack (replaced by the next candidate):
    L5 row 252: 

=== TF ===
  L1: 1/1 accepted ✅
  L2: 2/2 accepted ✅
  L3: 3/3 accepted ✅
  L4: 4/4 accepted ✅
  L5: 5/5 accepted ✅
  manual picks that failed on LocalStack (replaced by the next candidate):
    L1 row 319: 
    L2 row 56: 
    L4 row 148: 
    L4 row 199: 


## §16 Assemble and export the extended ablation benchmark

In [20]:
ext_final, ext_added = {}, {}
for lang in ('cfn', 'tf'):
    q = ext_queue[lang]
    add = q[q['accepted'] == True].copy()
    add['row_number'] = add['row_number'].astype(int)
    add_out = add[['row_number', 'path_key', 'user_prompt', 'difficulty']].rename(
        columns={'path_key': 'ground_truth_path', 'user_prompt': 'prompt'})
    final = pd.concat([base[lang][['row_number', 'ground_truth_path', 'prompt', 'difficulty']], add_out],
                      ignore_index=True).sort_values('row_number').reset_index(drop=True)
    assert not final['row_number'].duplicated().any(), f'{lang}: duplicate row_number'
    target = allocate_targets(EXTENDED_TARGET_PER_LANG, LEVEL_WEIGHTS)
    got = final['difficulty'].value_counts().to_dict()
    short = {l: target[l] - got.get(l, 0) for l in target if got.get(l, 0) < target[l]}
    print(f"{lang.upper()}: {len(base[lang])} original + {len(add_out)} added = {len(final)} "
          f"| per level {dict(sorted(got.items()))}" + (f' ⚠️ short {short}' if short else ' ✅'))
    ext_final[lang], ext_added[lang] = final, add_out.sort_values('row_number')

    sel = pd.concat([
        base[lang].assign(batch='v1', role='original', reason=''),
        add.assign(batch='v2')[['row_number', 'path_key', 'user_prompt', 'difficulty', 'role', 'reason',
                                 'batch', 'diversity_key', 'tier', 'dsv4f_genuine_iterations',
                                 'n_runs_hard', 'ext_source']].rename(
            columns={'path_key': 'ground_truth_path', 'user_prompt': 'prompt'}),
    ], ignore_index=True).assign(lang=lang)
    ext_final[lang + '_selection'] = sel

if EXT_EXPORT:
    for lang in ('cfn', 'tf'):
        for p in ABLATION_EVAL[lang]:
            ext_final[lang].to_csv(p, index=False)
        for p in (ADDITIONS_CSV[lang], ADDITIONS_CSV_IACGOD[lang]):
            ext_added[lang].to_csv(p, index=False)
        print(f'{lang.upper()}: {len(ext_final[lang])} rows -> {ABLATION_EVAL[lang][0].name} (+ IaCGOD copy); '
              f'{len(ext_added[lang])} additions -> {ADDITIONS_CSV[lang].name} (+ IaCGOD copy)')
    pd.concat([ext_final['cfn_selection'], ext_final['tf_selection']], ignore_index=True).to_csv(SELECTION_V2_CSV, index=False)
    print(f'Selection record -> {SELECTION_V2_CSV}')
else:
    print('EXT_EXPORT is False -- nothing written.')

CFN: 35 original + 15 added = 50 | per level {1: 3, 2: 7, 3: 10, 4: 13, 5: 17} ✅
TF: 35 original + 15 added = 50 | per level {1: 3, 2: 7, 3: 10, 4: 13, 5: 17} ✅
CFN: 50 rows -> cfn_eval_benchmark_ablation.csv (+ IaCGOD copy); 15 additions -> cfn_eval_benchmark_ablation_additions.csv (+ IaCGOD copy)
TF: 50 rows -> tf_eval_benchmark_ablation.csv (+ IaCGOD copy); 15 additions -> tf_eval_benchmark_ablation_additions.csv (+ IaCGOD copy)
Selection record -> /Users/iksena/Documents/research/data_analysis/ablation_selection/ablation_benchmark_selection_v2.csv


## §17 After extending